# ISIC 2024 - 最終提交（LightGBM ＋ 圖片模型特徵）

**Kaggle 設定**：
1. Input：比賽資料 `isic-2024-challenge`
2. Input：**`isic2024_image_models` 的輸出**（Add Input → Your Work）。加入哪些實驗版本就用哪些，會自動偵測
   （組員 E 平常只要加自己跑的 `BASE`；W15 最終提交時再加入全組最好的設定）
3. Accelerator **GPU**、Internet **關掉** → Save Version → Submit

流程：metadata 特徵 ＋ 病人內標準化 ＋ **每個圖片模型的預測當特徵** → LightGBM 5 折 → 測試集預測。
同時印出「只用 metadata」「加單一圖片模型」「加全部圖片模型」的 CV 比較。

In [ ]:
import glob, io, json, os, time
import numpy as np
import pandas as pd
import h5py
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

KAGGLE_INPUT = '/kaggle/input'

def find_dir(fname):
    # Kaggle 的掛載路徑會變（例如多一層 competitions/），所以用搜尋的找檔案所在資料夾
    for depth in range(1, 5):
        hits = sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, fname)),
                      key=lambda p: ('isic-2024-challenge' not in p, p))  # 優先選比賽本身的資料
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'在 {KAGGLE_INPUT} 底下找不到 {fname}，請確認右側 Input 有加入')

DATA = find_dir('train-metadata.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
os.makedirs('figures', exist_ok=True)
print('資料路徑:', DATA, '| device:', DEVICE)

In [ ]:
import lightgbm as lgb
NEG_RATIO = 50
CFG = dict(img=128, bs=128, workers=4)

# 自動找出所有圖片實驗的輸出（exp_{EXP}.json，kind == 'image'）
CNN_RUNS = {}
for depth in range(1, 5):
    for p in sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, 'exp_*.json'))):
        r = json.load(open(p, encoding='utf-8'))
        if r.get('kind') == 'image':
            CNN_RUNS.setdefault(r['exp'], (os.path.dirname(p), r['cfg'], r['oof_pauc']))
for k, (d, c, score) in CNN_RUNS.items():
    print(f"找到實驗 {k}（{c['model']}）: OOF pAUC {score:.4f}  ({d})")
if not CNN_RUNS:
    print('沒有找到圖片模型的輸出，只用 metadata')

In [ ]:
def pauc_above_tpr(y_true, y_pred, min_tpr=0.80):
    # 官方評分：ROC 曲線在 TPR >= 0.8 的面積，範圍 0 ~ 0.2
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = 1.0 - np.asarray(y_pred)
    max_fpr = 1 - min_tpr
    scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    return 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (scaled - 0.5)

def make_folds(train):
    # 跟 LightGBM、MLP 完全相同的切法，融合時才不會洩漏
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    return list(sgkf.split(train, train['target'], train['patient_id']))

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def make_tfms(img, aug='basic'):
    """aug: none（不增強）/ basic（翻轉、旋轉、色彩）/ strong（再加隨機裁切縮放、仿射、模糊）"""
    norm = [T.ToTensor(), T.Normalize(MEAN, STD)]
    if aug == 'none':
        train_ops = [T.Resize((img, img))]
    elif aug == 'basic':
        train_ops = [T.Resize((img, img)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                     T.RandomRotation(90), T.ColorJitter(0.2, 0.2, 0.1, 0.02)]
    elif aug == 'strong':
        train_ops = [T.RandomResizedCrop(img, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                     T.RandomRotation(180), T.RandomAffine(0, translate=(0.1, 0.1), shear=10),
                     T.ColorJitter(0.3, 0.3, 0.2, 0.05), T.RandomApply([T.GaussianBlur(3)], p=0.3)]
    else:
        raise ValueError(aug)
    eval_tfm = T.Compose([T.Resize((img, img))] + norm)
    return T.Compose(train_ops + norm), eval_tfm


class LesionDataset(Dataset):
    """從 hdf5 讀 JPEG bytes。每個 worker 各自開檔，避免 h5py 多程序問題。meta 是選用的表格特徵。"""
    def __init__(self, h5_path, ids, targets=None, tfm=None, meta=None):
        self.h5_path, self.ids, self.targets, self.tfm, self.meta = h5_path, list(ids), targets, tfm, meta
        self.h5 = None
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, 'r')
        img = Image.open(io.BytesIO(bytes(self.h5[self.ids[i]][()]))).convert('RGB')
        x = self.tfm(img)
        y = torch.tensor(-1.0 if self.targets is None else float(self.targets[i]))
        m = torch.zeros(0) if self.meta is None else torch.from_numpy(self.meta[i])
        return x, m, y


@torch.no_grad()
def predict(model, h5_path, ids, tfm, meta=None):
    dl = DataLoader(LesionDataset(h5_path, ids, tfm=tfm, meta=meta), batch_size=CFG['bs'] * 2,
                    shuffle=False, num_workers=CFG['workers'])
    model.eval()  # BatchNorm 用累積的統計量、Dropout 關閉
    out = []
    for x, m, _ in dl:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            logit = model(x.to(DEVICE), m.to(DEVICE))
        out.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

In [ ]:
class SimpleCNN(nn.Module):
    """自己搭的 CNN（對應上課 W5）：[Conv3x3 → BN → ReLU] ×2 → MaxPool，重複 4 次 → GAP → Dropout → FC"""
    def __init__(self, n_out=1):
        super().__init__()
        layers, c_in = [], 3
        for c in [32, 64, 128, 256]:
            layers += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(c_in, n_out))
        self.out_dim = c_in
    def forward_features(self, x):
        return self.pool(self.features(x)).flatten(1)
    def forward(self, x):
        return self.head(self.pool(self.features(x)))


def build_backbone(name, pretrained):
    """回傳 (去掉分類頭的骨幹, 特徵維度)。圖片分類模型和多模態模型共用。"""
    if name == 'simple_cnn':
        m = SimpleCNN()
        m.forward = m.forward_features
        return m, m.out_dim
    if name == 'resnet18':
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        d = m.fc.in_features; m.fc = nn.Identity()
        return m, d
    if name == 'efficientnet_b0':
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        d = m.classifier[1].in_features; m.classifier = nn.Identity()
        return m, d
    if name == 'vit_tiny':
        import timm  # Kaggle 內建；Vision Transformer（對應上課 W8）
        m = timm.create_model('vit_tiny_patch16_224', pretrained=pretrained, num_classes=0, img_size=CFG['img'])
        return m, m.num_features
    raise ValueError(name)


class ImageClassifier(nn.Module):
    def __init__(self, name, pretrained, dropout=0.3):
        super().__init__()
        self.backbone, d = build_backbone(name, pretrained)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, 1))
    def forward(self, x, meta=None):
        return self.head(self.backbone(x)).squeeze(1)

## 讀資料、建 metadata 特徵

In [ ]:
train = pd.read_csv(f'{DATA}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA}/test-metadata.csv')
drop = {'isic_id', 'patient_id', 'target', 'image_type', 'attribution', 'copyright_license'}
common = [c for c in test.columns if c in train.columns and c not in drop]
cat_cols = [c for c in common if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in common if c not in cat_cols]
for c in cat_cols:
    cats = pd.Categorical(train[c]).categories
    train[c] = pd.Categorical(train[c], categories=cats)
    test[c] = pd.Categorical(test[c], categories=cats)
pn_cols = [c for c in num_cols if c not in ('age_approx', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z')]
def add_patient_feats(df):
    g = df.groupby('patient_id')[pn_cols]
    z = (df[pn_cols] - g.transform('mean')) / (g.transform('std') + 1e-6)
    z.columns = [c + '_pnorm' for c in pn_cols]
    df = pd.concat([df, z], axis=1)
    df['patient_n_lesions'] = df.groupby('patient_id')['isic_id'].transform('count')
    return df
train, test = add_patient_feats(train), add_patient_feats(test)
base_features = num_cols + cat_cols + [c + '_pnorm' for c in pn_cols] + ['patient_n_lesions']

## 圖片模型的預測當特徵（train 用 OOF，test 用 5 個 fold 模型平均）

In [ ]:
img_features = []
for name, (d, cfg, _) in CNN_RUNS.items():
    CFG['img'] = cfg['img']
    _, eval_tfm = make_tfms(cfg['img'])
    col = f'img_{name}'
    train = train.merge(pd.read_csv(f'{d}/oof_{name}.csv').rename(columns={'pred': col}), on='isic_id', how='left')
    p = np.zeros(len(test))
    for fold in range(5):
        m = ImageClassifier(cfg['model'], False, cfg['dropout']).to(DEVICE)
        m.load_state_dict(torch.load(f'{d}/{name}_fold{fold}.pt', map_location=DEVICE))
        p += predict(m, f'{DATA}/test-image.hdf5', test.isic_id.values, eval_tfm) / 5
    test[col] = p
    for df in (train, test):
        df[col + '_prank'] = df.groupby('patient_id')[col].rank(pct=True)  # 這顆是不是這個人最可疑的
    img_features += [col, col + '_prank']
print('圖片特徵：', img_features)

## LightGBM 5 折：比較有無圖片特徵

In [ ]:
params = dict(objective='binary', learning_rate=0.03, num_leaves=15, min_child_samples=40,
              colsample_bytree=0.5, subsample=0.8, subsample_freq=1, reg_lambda=5,
              n_estimators=200, random_state=SEED, verbose=-1)
folds = make_folds(train)

def run_cv(feats):
    oof, models_ = np.zeros(len(train)), []
    for tr, va in folds:
        d = train.iloc[tr]
        pos = d[d.target == 1]
        neg = d[d.target == 0].sample(min(len(pos) * NEG_RATIO, int((d.target == 0).sum())), random_state=SEED)
        d = pd.concat([pos, neg])
        m = lgb.LGBMClassifier(**params).fit(d[feats], d.target)
        oof[va] = m.predict_proba(train.iloc[va][feats])[:, 1]
        models_.append(m)
    return pauc_above_tpr(train.target, oof), models_

results = {}
results['LightGBM（metadata）'], models_final = run_cv(base_features)
features = base_features
if CNN_RUNS:
    for name in CNN_RUNS:
        results[f'LightGBM + {name}'], _ = run_cv(base_features + [f'img_{name}', f'img_{name}_prank'])
    features = base_features + img_features
    results['LightGBM + 全部圖片模型'], models_final = run_cv(features)
for k, v in results.items():
    print(f'{k:36s} OOF pAUC = {v:.4f}')
pd.Series(results, name='oof_pauc').to_csv('fusion_results.csv')

## 產生 submission.csv

In [ ]:
pred = np.mean([m.predict_proba(test[features])[:, 1] for m in models_final], axis=0)
sub = pd.DataFrame({'isic_id': test['isic_id'], 'target': pred})
sub.to_csv('submission.csv', index=False)
sub.head()